In [0]:
from azure.storage.blob import BlobClient, ContainerClient
import pandas as pd
from io import BytesIO

connection_string = (
    "DefaultEndpointsProtocol=https;"
    "AccountName=;"
    "AccountKey=")

# Nome do container
container_name = "azuredatalake"

# Cria o client do container
container_client = ContainerClient.from_connection_string(
    conn_str=connection_string,
    container_name=container_name
)

directory_prefix = 'teste/'
blobs = container_client.list_blobs(name_starts_with=directory_prefix)

for file in blobs:
   
    blob_client = BlobClient.from_connection_string(
        conn_str=connection_string,
        container_name="azuredatalake",
        blob_name=file.name
    )

    streamdownloader = blob_client.download_blob()
    conteudo_bytes = streamdownloader.readall()

    buffer_txt = BytesIO(conteudo_bytes)

    df_pandas = pd.read_parquet(buffer_txt)
    df_pandas.columns = [col.strip().replace(' ', '_').replace('.', '_').replace('-', '_') for col in df_pandas.columns]
    df = spark.createDataFrame(df_pandas)

    name = str(file.name) 
    print(name); name = name.split('/')[1].replace('-','_').replace('.parquet','')
    print(name)
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f'ted_dev.dev_moises_santos.{name}')